# TTC Subway Delay Analysis
Average and maximum delay per month, 2025 YTD.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import polars as pl
import altair as alt
from sixsight.transforms.ttc_subway_delay import monthly_stats

In [ ]:
CSV_PATH = Path("../data/raw/ttc-subway-delay-data/ttc-subway-delay-data-since-2025.csv")

df = pl.read_csv(CSV_PATH, try_parse_dates=False)
print(df.schema)
df.head(3)

In [3]:
monthly = monthly_stats(df)
monthly

year_month,month_label,total_delay,max_delay
i32,str,i64,i64
202501,"""Jan 2025""",6381,360
202502,"""Feb 2025""",8766,900
202503,"""Mar 2025""",6305,140
202504,"""Apr 2025""",6055,202
202505,"""May 2025""",5432,143
…,…,…,…
202509,"""Sep 2025""",4179,149
202510,"""Oct 2025""",5150,76
202511,"""Nov 2025""",6187,480


In [4]:
month_order = monthly["month_label"].to_list()

base = alt.Chart(monthly).encode(
    x=alt.X("month_label:N", sort=month_order, title="Month"),
)

chart_total = base.transform_calculate(
    total_delay_days="datum.total_delay / 1440"
).mark_bar(color="#4C72B0").encode(
    y=alt.Y("total_delay_days:Q", title="Total Delay (days)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("total_delay_days:Q", title="Total Delay (days)", format=".2f"),
    ],
).properties(title="Total Delay per Month", width=650, height=300)

chart_max = base.transform_calculate(
    max_delay_hours="datum.max_delay / 60"
).mark_bar(color="#DD8452").encode(
    y=alt.Y("max_delay_hours:Q", title="Max Delay (hours)"),
    tooltip=[
        alt.Tooltip("month_label:N", title="Month"),
        alt.Tooltip("max_delay_hours:Q", title="Max Delay (hours)", format=".1f"),
    ],
).properties(title="Max Delay per Month", width=650, height=300)

alt.vconcat(chart_total, chart_max)

alt.VConcatChart(...)